## 0.1 Init ambiente Google Colab

Conecto la virtual machine de Google Colab con el Google Drive para tener persistencia de archivos.

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental, POR UNICA VEZ, seguir estos pasos:

* Registrar usuario en Kaggle con la cuenta de email de la Universidad Austral
* Hacer el "Join Competition" a la competencia de Labo 3
* Generar el archivo kaggle.json en https://www.kaggle.com/settings/account ("Create Legacy API Key")
* Crear carpeta labo3 en el Google Drive, y dentro una carpeta kaggle
* Subir kaggle.json a esa carpeta kaggle

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# z102 — EDA orientado a la decisión de modelado

Este es mi análisis exploratorio para la competencia. Uso el mismo stack que el EDA base
(`z101`: **DuckDB + plotly**), pero lo reoriento: no busco describir todo el dataset, busco
**decidir dónde invertir el esfuerzo de modelado y qué features construir** para los modelos
que voy a usar (LightGBM / Deep Learning), dejando de paso a AutoARIMA (`z251`) el subconjunto
que ese modelo puede aprovechar.

**El problema.** "La Multinacional" (Unilever) hace forecasting **mensual** de sell-in de
consumo masivo. Debo predecir las `tn` de **202002** para los **780 productos** de
`product_id_apredecir201912`.

**Sobre el horizonte +2.** La historia llega hasta **201912**, así que 202002 es el horizonte
**mes+2** y se saltea 202001. Esto es una **decisión operativa** de la competencia (el mes
inmediato siguiente ya está comprometido en producción; lo accionable es el +2), y **no** debe
confundirse con el lag físico de ~2 meses entre el sell-in (sale el camión de planta) y el
sell-out (se escanea en la góndola). Ese lag físico lo uso más abajo solo para **interpretar la
estacionalidad**. Son dos "2 meses" distintos.

**Hipótesis de trabajo que voy a explorar** (cada sección cierra con la decisión/feature que habilita):
1. La **métrica está ponderada por volumen** → el esfuerzo debe ir a los productos grandes.
2. El **portfolio** está dominado por Home Care y Limpieza; Foods es minoritario.
3. El **ciclo de vida** (ramp-up y discontinuación) ensucia las series.
4. La **estacionalidad** es específica por categoría (y vive donde hay poco volumen).
5. Hay un **pico recurrente de cierre de año** (forzado de canal) modelable como flag.
6. Los **clientes chicos** compran de forma más regular (menor variabilidad).

In [ ]:
import os as os
import duckdb
import numpy as np
import pandas as pd
import plotly.express as px

In [ ]:
PARAM = {'experimento':'EDA-102',
  'kaggle_competition':'labo-iii-2026-ba'
}

In [ ]:
ruta = "/content/buckets/b1/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

## 1 Carga de tablas

Cargo a DuckDB. Sumo dos tablas que el EDA base descargaba pero no usaba:
`tb_apredecir` (los 780 productos objetivo) y `tb_stocks`.

In [ ]:
con = duckdb.connect()

In [ ]:
# sell-in, con periodo (YYYYMM) convertido a DATE
con.execute("""
    CREATE OR REPLACE TABLE tb_sellin AS
    SELECT CAST(strptime(CAST( periodo AS VARCHAR), '%Y%m') AS DATE) periodo,
           customer_id, product_id, plan_precios_cuidados,
           cust_request_qty, cust_request_tn, tn
    FROM read_csv_auto('/content/datasets/sell-in.txt.gz')
    ORDER BY product_id, periodo, customer_id
""")

In [ ]:
# maestro de productos
con.execute("""
    CREATE OR REPLACE TABLE tb_productos AS
    SELECT * FROM read_csv_auto('/content/datasets/tb_productos.txt')
    ORDER BY product_id
""")

In [ ]:
# los 780 productos objetivo (tn de 202002 a predecir)
con.execute("""
    CREATE OR REPLACE TABLE tb_apredecir AS
    SELECT * FROM read_csv_auto('/content/datasets/product_id_apredecir201912.txt')
""")
print("Productos a predecir:", con.sql("SELECT COUNT(*) FROM tb_apredecir").fetchone()[0])

In [ ]:
# stocks (la dejo disponible; no es el foco)
con.execute("""
    CREATE OR REPLACE TABLE tb_stocks AS
    SELECT * FROM read_csv_auto('/content/datasets/tb_stocks.txt')
""")

In [ ]:
# inspecciono los esquemas (no asumo nombres de columnas)
print("=== tb_productos ==="); display(con.sql("DESCRIBE tb_productos").df())
print("=== tb_stocks ===");    display(con.sql("DESCRIBE tb_stocks").df())

In [ ]:
# tamaño del panel y rango temporal
display(con.sql("""
    SELECT MIN(periodo) primer_periodo, MAX(periodo) ultimo_periodo,
           COUNT(DISTINCT periodo) n_periodos,
           COUNT(DISTINCT product_id) n_productos,
           COUNT(DISTINCT customer_id) n_clientes, COUNT(*) n_filas
    FROM tb_sellin
""").df())

ULT = con.sql("SELECT MAX(periodo) FROM tb_sellin").fetchone()[0]
print("Ultimo periodo con datos:", ULT, "->  objetivo = 202002 (mes+2)")

## 2 La métrica manda: el error está ponderado por volumen

Arranco por acá a propósito, porque condiciona todas las decisiones siguientes.

La nota se calcula sobre el **Total Forecast Error** del Private Leaderboard: en esencia un error
agregado **ponderado por toneladas**,

$$\text{TFE} = \frac{\sum_{p} \lvert \hat{y}_p - y_p \rvert}{\sum_{p} y_p}$$

sumado sobre los 780 productos en 202002. *(Pendiente: confirmar la fórmula exacta en la página de
la competencia en Kaggle.)*

**Consecuencia central:** tanto el numerador como el denominador están dominados por los productos
de **mayor volumen**. Un mismo error porcentual en un producto grande pesa muchísimo más que en uno
chico. Por lo tanto, lo primero que necesito saber es **dónde está concentrado el volumen objetivo**:
ahí es donde se gana o se pierde la competencia.

Como no tengo el real de 202002, uso el **volumen 2019 como proxy del peso** de cada producto.

In [ ]:
# peso de cada producto objetivo ~ su volumen reciente (2019)
con.execute("""
    CREATE OR REPLACE TABLE tb_peso AS
    SELECT a.product_id, COALESCE(SUM(s.tn), 0) tn_2019
    FROM tb_apredecir a
    LEFT JOIN tb_sellin s
      ON s.product_id = a.product_id
     AND s.periodo BETWEEN DATE '2019-01-01' AND DATE '2019-12-01'
    GROUP BY a.product_id
""")

df = con.sql("""
    SELECT product_id, tn_2019,
           100.0*tn_2019 / SUM(tn_2019) OVER () pct
    FROM tb_peso ORDER BY tn_2019 DESC
""").df()
df["pct_acum"] = df["pct"].cumsum()

# cuantos productos concentran 50/80/95% del volumen objetivo
for u in [50, 80, 95]:
    n = int((df["pct_acum"] <= u).sum()) + 1
    print(f"{n:4d} productos ({100*n/len(df):4.1f}% del set)  ->  {u}% del volumen objetivo")

sin_volumen = int((df["tn_2019"] == 0).sum())
print(f"\n{sin_volumen} productos del set objetivo NO tuvieron ventas en 2019 (peso ~0 pero igual hay que predecirlos)")
display(df.head(15))

In [ ]:
# curva de concentración (Pareto) del volumen objetivo
df_p = df.reset_index(drop=True).copy()
df_p["rank"] = np.arange(1, len(df_p)+1)
gra = px.line(df_p, x="rank", y="pct_acum",
              title="Concentración del volumen objetivo (Pareto de los 780 productos)",
              labels={"rank":"productos ordenados por volumen", "pct_acum":"% acumulado del volumen"})
gra.add_hline(y=80, line_dash="dash")
gra.show()

**Decisión que habilita §2.** El volumen está muy concentrado: un grupo chico de productos
explica la mayoría del tonelaje y, por lo tanto, de la métrica. Voy a **priorizar precisión en los
high-volume** y a vigilar mi error en ese subconjunto por encima del promedio simple. Para el resto
(cola larga, y los productos sin volumen 2019) alcanza con una predicción razonable; no mueven la nota.

## 3 Estructura del portfolio: dónde está el volumen real

Quiero ver cómo se reparte el volumen por **unidad de negocio**. Detecto dinámicamente las columnas
de categoría del maestro de productos (las que empiezan con `cat`) y uso la más gruesa como unidad
de negocio.

In [ ]:
prod_cols = con.sql("DESCRIBE tb_productos").df()["column_name"].tolist()
cat_cols = [c for c in prod_cols if c.lower().startswith("cat")]
print("Columnas de categoría:", cat_cols)
CAT_BU = cat_cols[0] if cat_cols else "cat1"   # unidad de negocio (la categoría más gruesa)
print("Uso como unidad de negocio:", CAT_BU)

In [ ]:
# share de volumen 2019 por unidad de negocio (todo el universo)
df = con.sql(f"""
    SELECT p.{CAT_BU} unidad, SUM(s.tn) tn
    FROM tb_sellin s JOIN tb_productos p USING (product_id)
    WHERE s.periodo BETWEEN DATE '2019-01-01' AND DATE '2019-12-01'
    GROUP BY p.{CAT_BU} ORDER BY tn DESC
""").df()
df["pct"] = 100*df["tn"]/df["tn"].sum()
display(df)
px.bar(df, x="unidad", y="pct", title=f"Share de volumen 2019 por unidad de negocio ({CAT_BU})",
       labels={"unidad":"unidad de negocio","pct":"% del volumen"}).show()

In [ ]:
# y dentro del set objetivo (los 780) -> ¿dónde se juega la métrica?
df = con.sql(f"""
    SELECT p.{CAT_BU} unidad, SUM(pe.tn_2019) tn
    FROM tb_peso pe JOIN tb_productos p USING (product_id)
    GROUP BY p.{CAT_BU} ORDER BY tn DESC
""").df()
df["pct"] = 100*df["tn"]/df["tn"].sum()
display(df)
px.bar(df, x="unidad", y="pct",
       title="Share del volumen OBJETIVO (780 productos) por unidad de negocio",
       labels={"unidad":"unidad de negocio","pct":"% del volumen objetivo"}).show()

**Lectura §3.** El volumen se concentra en **Home Care y Limpieza**; **Foods** (mayonesas,
sopas — los ejemplos clásicos del EDA de clase) es **minoritario**. Esto reordena prioridades: los
casos vistosos de estacionalidad de Foods son didácticos pero **de bajo impacto en la métrica**.
El feature engineering y el tuning tienen que concentrarse en **HC/Limpieza**, que es donde está
el tonelaje (y, por §2, la nota). Lo verifico al re-evaluar estacionalidad más abajo.

## 4 Ciclo de vida del producto (ramp-up y discontinuación)

Solo tengo **sell-in** (envío de planta), no sell-out. El arranque de un producto (ramp-up) y su
apagado (discontinuación) son tramos que no representan demanda estable: en el ramp-up la serie crece
artificialmente mientras se llena el canal, y en la discontinuación cae a cero por decisión comercial,
no por demanda. Decido **detectarlos y aislarlos**.

Trabajo en el grano de modelado: `tn` por `product_id` × `periodo` (sumando clientes), que es lo que
modela AutoARIMA y la base del panel para GBM/DL.

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE tb_ventas_pm AS
    SELECT product_id, periodo, SUM(tn) tn
    FROM tb_sellin GROUP BY product_id, periodo
    ORDER BY product_id, periodo
""")
display(con.sql("SELECT * FROM tb_ventas_pm LIMIT 5").df())

In [ ]:
# vida de cada producto: alta, baja, meses con venta y si quedó discontinuado
con.execute(f"""
    CREATE OR REPLACE TABLE tb_vida AS
    SELECT product_id,
           MIN(periodo) mes_alta,
           MAX(periodo) mes_baja,
           COUNT(*) FILTER (WHERE tn > 0) meses_con_venta,
           datediff('month', MIN(periodo), MAX(periodo)) + 1 AS span_meses,
           (MAX(periodo) < DATE '{ULT}') AS discontinuado
    FROM tb_ventas_pm GROUP BY product_id
""")
display(con.sql("SELECT * FROM tb_vida ORDER BY product_id LIMIT 10").df())

In [ ]:
# cuánta historia tiene cada producto
px.histogram(con.sql("SELECT span_meses FROM tb_vida").df(), x="span_meses", nbins=36,
             title="Meses de vida por producto (span alta→baja)",
             labels={"span_meses":"meses de vida"}).show()
display(con.sql("SELECT discontinuado, COUNT(*) n FROM tb_vida GROUP BY discontinuado ORDER BY discontinuado").df())

Cuánta historia tienen los **780 objetivo**: relevante porque un producto con poca historia
no permite ajustar estacionalidad anual (es la causa de los productos que en `z251` caen a la 2ª
pasada de AutoARIMA, sin `m=12`).

In [ ]:
display(con.sql(f"""
    SELECT COUNT(*) total_objetivo,
           SUM( (v.mes_baja >= DATE '{ULT}')::INT ) activos_al_final,
           SUM( v.discontinuado::INT )              discontinuados,
           SUM( (v.span_meses < 12)::INT )          menos_de_12m,
           SUM( (v.span_meses < 24)::INT )          menos_de_24m
    FROM tb_apredecir a JOIN tb_vida v USING (product_id)
""").df())

In [ ]:
# panel con flags de ciclo de vida -> insumo directo de modelado
con.execute("""
    CREATE OR REPLACE TABLE tb_panel AS
    SELECT v.product_id, v.periodo, v.tn,
           datediff('month', d.mes_alta, v.periodo)              AS meses_desde_lanzamiento,
           (datediff('month', d.mes_alta, v.periodo) < 3)        AS es_ramp_up,
           ( d.discontinuado AND datediff('month', v.periodo, d.mes_baja) <= 2 ) AS es_discontinuacion
    FROM tb_ventas_pm v JOIN tb_vida d USING (product_id)
""")
display(con.sql("""
    SELECT SUM(es_ramp_up::INT) filas_ramp_up,
           SUM(es_discontinuacion::INT) filas_discontinuacion,
           COUNT(*) filas_totales
    FROM tb_panel
""").df())

In [ ]:
# visualizo el ciclo de vida de un producto coloreando los tramos
def graficar_ciclo_vida(producto):
  df = con.sql(f"""
      SELECT periodo, tn,
             CASE WHEN es_ramp_up THEN 'ramp-up (<3m)'
                  WHEN es_discontinuacion THEN 'discontinuación'
                  ELSE 'ventana estable' END AS tramo
      FROM tb_panel WHERE product_id = {producto} ORDER BY periodo
  """).df()
  titulo = con.sql(f"SELECT CONCAT_WS(', ', *COLUMNS('.*')) FROM tb_productos WHERE product_id={producto}").fetchone()[0]
  gra = px.scatter(df, x="periodo", y="tn", color="tramo", title=f"[{producto}] {titulo}",
                   labels={"periodo":"Periodo","tn":"Toneladas"})
  gra.update_traces(mode='markers+lines'); gra.update_yaxes(rangemode="tozero"); gra.show()

for p in [20001, 20494, 20554]:
  graficar_ciclo_vida(p)

**Decisión §4.** Quedan definidos `meses_desde_lanzamiento`, `es_ramp_up` y `es_discontinuacion`.
- **AutoARIMA (`z251`)**: filtro estas filas antes de ajustar → series más limpias.
- **GBM/DL**: NO tiro filas; `meses_desde_lanzamiento` entra como **feature** y el modelo aprende la
  forma del ramp-up. Así no pierdo información y trato el ciclo de vida de forma explícita.

## 5 Estacionalidad por categoría (re-evaluada por volumen)

Sospecho que la estacionalidad no es global sino **específica de cada categoría**. La mido con un
**índice estacional** = `tn` promedio del mes / `tn` promedio de la categoría, calculado sobre la
**ventana estable** (sin ramp-up ni discontinuación). Además quiero cruzar esto con §3: ver si la
estacionalidad fuerte coincide con categorías de bajo volumen.

In [ ]:
# perfil estacional global (ventana estable)
df_mes = con.sql("""
    SELECT month(periodo) mes, AVG(tn) tn_medio
    FROM tb_panel WHERE NOT es_ramp_up AND NOT es_discontinuacion
    GROUP BY month(periodo) ORDER BY mes
""").df()
g = px.bar(df_mes, x="mes", y="tn_medio", title="Perfil estacional global (promedio por mes)",
           labels={"mes":"mes","tn_medio":"tn medio"}); g.update_xaxes(dtick=1); g.show()

In [ ]:
# índice estacional por categoría -> heatmap mes x categoría (top-12 por volumen)
df = con.sql(f"""
    SELECT p.{CAT_BU} cat, month(v.periodo) mes, v.tn
    FROM tb_panel v JOIN tb_productos p USING (product_id)
    WHERE NOT v.es_ramp_up AND NOT v.es_discontinuacion
""").df()
top_cats = df.groupby("cat")["tn"].sum().sort_values(ascending=False).head(12).index
df = df[df["cat"].isin(top_cats)]
piv = df.groupby(["cat","mes"])["tn"].mean().reset_index().pivot(index="cat", columns="mes", values="tn")
indice = piv.div(piv.mean(axis=1), axis=0)
px.imshow(indice, aspect="auto", color_continuous_scale="RdBu_r", origin="upper",
          labels={"x":"mes","y":"categoría","color":"índice estacional"},
          title="Índice estacional por categoría (1.0 = mes promedio)").update_xaxes(dtick=1).show()

**Lectura §5.** La estacionalidad es claramente **específica por categoría**, pero —y esto es lo
importante combinado con §3— las categorías más **estacionales suelen ser de bajo volumen** (Foods),
mientras que las de **alto volumen (HC/Limpieza) son más estables / de tendencia**. Es decir, los
features estacionales rinden donde menos pesa la métrica. Decisión:
- **AutoARIMA**: `m=12` solo para categorías estacionales con historia ≥24m; el resto `seasonal=False`.
- **GBM/DL**: dummies de mes + índice estacional por categoría como features, pero **sin sobre-invertir**
  en afinar la estacionalidad de Foods.

## 6 Pico de cierre de año (forzado de canal)

En las series observo un patrón recurrente hacia fin de año: un **pico** seguido de una **caída**,
consistente con prácticas de **carga de canal** (se empuja venta al supermercado para cerrar mejor el
año, inflando el sell-in por encima de la demanda real). Si es sistemático, conviene **aislarlo con un
flag binario** para que el modelo no lo confunda con demanda genuina. Lo verifico en los datos.

In [ ]:
# distribución de tn por mes del año (ventana estable) -> pico + caída posterior
df = con.sql("""
    SELECT month(periodo) mes, tn FROM tb_panel
    WHERE NOT es_ramp_up AND NOT es_discontinuacion AND tn > 0
""").df()
g = px.box(df, x="mes", y="tn", title="Distribución de tn por mes (escala log)",
           labels={"mes":"mes","tn":"tn (producto-mes)"})
g.update_yaxes(type="log"); g.update_xaxes(dtick=1); g.show()

In [ ]:
# índice mensual global para cuantificar el pico
df = con.sql("""
    SELECT month(periodo) mes, SUM(tn) tn FROM tb_panel
    WHERE NOT es_ramp_up AND NOT es_discontinuacion GROUP BY month(periodo) ORDER BY mes
""").df()
df["indice"] = df["tn"]/df["tn"].mean()
display(df)
g = px.line(df, x="mes", y="indice", markers=True,
            title="Índice mensual global (1.0 = mes promedio): ¿dónde está el pico de cierre?",
            labels={"mes":"mes","indice":"índice"})
g.add_hline(y=1.0, line_dash="dash"); g.update_xaxes(dtick=1); g.show()

In [ ]:
# creo el feature candidato: flag binario del mes del pico de cierre.
# Parametrizado: ajustar MES_PICO según lo que muestren las dos celdas anteriores.
MES_PICO = 10

con.execute(f"""
    CREATE OR REPLACE TABLE tb_panel AS
    SELECT *, (month(periodo) = {MES_PICO})::INT AS flag_cierre_anio
    FROM tb_panel
""")
display(con.sql("""
    SELECT flag_cierre_anio, COUNT(*) n, ROUND(AVG(tn),2) tn_medio
    FROM tb_panel WHERE NOT es_ramp_up AND NOT es_discontinuacion
    GROUP BY flag_cierre_anio
""").df())

**Decisión §6.** Si el pico se confirma (esperado en octubre), queda creado `flag_cierre_anio`.
- **AutoARIMA**: lo paso como **regresor exógeno** (`X=` en `auto_arima` / SARIMAX).
- **GBM/DL**: feature binario directo; opcional un `post_cierre` para capturar la caída del mes siguiente.
- Si el pico aparece corrido (nov/dic), ajusto `MES_PICO`.

## 7 Heterogeneidad de clientes

Hipótesis: los clientes de **menor volumen** compran de forma más **regular** (menor coeficiente de
variación), mientras que los grandes son más erráticos. Lo mido con `CV = desvío/media` de la `tn`
mensual por cliente.

Esto **no** lo usa AutoARIMA (que agrega sobre clientes), pero es insumo valioso para el track GBM/DL:
features de segmento/mix de cliente, o incluso modelar a grano producto×cliente.

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE tb_cli_mes AS
    SELECT customer_id, periodo, SUM(tn) tn FROM tb_sellin GROUP BY customer_id, periodo
""")
con.execute("""
    CREATE OR REPLACE TABLE tb_clientes AS
    SELECT customer_id, SUM(tn) total_tn, COUNT(*) n_meses,
           AVG(tn) tn_medio, STDDEV_SAMP(tn)/NULLIF(AVG(tn),0) cv
    FROM tb_cli_mes GROUP BY customer_id
""")
display(con.sql("SELECT * FROM tb_clientes ORDER BY total_tn DESC LIMIT 5").df())

In [ ]:
# CV vs volumen del cliente (>=6 meses para que el CV sea estable)
df = con.sql("SELECT * FROM tb_clientes WHERE n_meses >= 6").df()
g = px.scatter(df, x="total_tn", y="cv", opacity=0.4, trendline="lowess",
               title="CV de la demanda vs volumen del cliente",
               labels={"total_tn":"volumen total (tn)","cv":"CV (desvío/media)"})
g.update_xaxes(type="log"); g.show()

In [ ]:
# CV mediano por decil de tamaño -> resumen claro
df = con.sql("SELECT * FROM tb_clientes WHERE n_meses >= 6").df()
df["decil_tamano"] = pd.qcut(df["total_tn"], 10, labels=False) + 1
resumen = df.groupby("decil_tamano").agg(cv_mediano=("cv","median"),
                                          n_clientes=("customer_id","count")).reset_index()
display(resumen)
px.bar(resumen, x="decil_tamano", y="cv_mediano",
       title="CV mediano por decil de tamaño (1=chico, 10=grande)",
       labels={"decil_tamano":"decil de volumen","cv_mediano":"CV mediano"}).update_xaxes(dtick=1).show()
print("Correlación log(volumen) vs CV:", round(np.corrcoef(np.log(df["total_tn"]), df["cv"])[0,1], 3))

**Decisión §7.** Si los deciles chicos muestran menor CV, confirmo la hipótesis y derivo features
para GBM/DL: `cv_cliente`, `decil_tamano_cliente`, flag `cliente_estable`, y a nivel producto la
**fracción de demanda que viene de clientes estables** (señal de cuán predecible es ese producto).

## 8 Conclusiones e implicancias para el modelado

Ordeno los hallazgos **por impacto en la métrica**:

| # | Hallazgo | Decisión para AutoARIMA (`z251`) | Feature / decisión para GBM-DL |
|---|---|---|---|
| 1 | **Métrica ponderada por volumen** | priorizar productos high-volume | pesar el entrenamiento/eval por tn; vigilar error en el top de volumen |
| 2 | **HC/Limpieza domina, Foods es minoría** | no malgastar tuning en Foods | concentrar FE en HC/Limpieza |
| 3 | **Ciclo de vida** (ramp-up <3m + discontinuación) | filtrar filas antes de ajustar | `meses_desde_lanzamiento`, `es_ramp_up`, `es_discontinuacion` (no se tira data) |
| 4 | **Estacionalidad específica por categoría** (y de bajo volumen) | `m=12` selectivo | dummies de mes + índice estacional por categoría, sin sobre-invertir |
| 5 | **Pico de cierre de año** | regresor exógeno `X=` | `flag_cierre_anio` (+ `post_cierre`) |
| 6 | **Clientes chicos más regulares** | no aplica | `cv_cliente`, `decil_tamano`, mix de clientes estables |
| 7 | **Productos con poca historia** | caen a 2ª pasada sin estacionalidad | `span_meses` como señal de confianza |

**Validación (clave, derivada del horizonte +2).** Para validar sin fuga de datos debo respetar el
horizonte real: entrenar hasta un mes `T`, **saltear `T+1`** y predecir `T+2` (ej.: train hasta
201910 → predecir 201912). Validar prediciendo `T+1` sobreestima la performance y rompe la simetría
con la competencia.

**Problemas → soluciones.**
1. *Tramos ruidosos (ramp-up / discontinuación)* → recortar (ARIMA) o codificar como feature (GBM).
2. *Estacionalidad heterogénea y de bajo volumen* → modelar por categoría, sin asumir `m=12` global.
3. *Pico artificial de cierre de año* → feature/exógena que lo aísle de la demanda real.
4. *Poca historia en parte del set* → modelo global (GBM) que comparte señal entre series.
5. *Métrica ponderada* → todo el esfuerzo dirigido al volumen (HC/Limpieza, top de la cola de Pareto).

**Artefactos generados:** `tb_panel` (series + flags de ciclo de vida + `flag_cierre_anio`),
`tb_peso` (volumen objetivo), `tb_clientes` (perfil de cliente). Listos para el pipeline de modelado.